# Data Preprocessing: Sentinel-2 & GEDI L2A on Google Earth Engine

Visualize Sentinel-2 spectral imagery and NASA GEDI L2A canopy height values
over the state of California using `geemap`.

In [2]:
import ee
import geemap

ee.Authenticate()


Successfully saved authorization token.


In [3]:
ee.Initialize(project="canopy-height-ml")  # TODO: replace with your GEE project ID

## Define California AOI

In [4]:
california = (
    ee.FeatureCollection("TIGER/2018/States")
    .filter(ee.Filter.eq("NAME", "California"))
    .geometry()
)

## Sentinel-2 Surface Reflectance (latest cloud-free composite)

In [5]:
def mask_s2_clouds(image):
    """Mask clouds using the SCL band from Sentinel-2 SR."""
    scl = image.select("SCL")
    # Keep vegetation, bare soil, water — mask cloud & shadow classes
    mask = scl.neq(3).And(scl.neq(8)).And(scl.neq(9)).And(scl.neq(10))
    return image.updateMask(mask)


# Recent 90-day composite for California (today = 2026-04-09)
s2 = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterBounds(california)
    .filterDate("2026-01-09", "2026-04-09")
    .map(mask_s2_clouds)
    .median()
    .clip(california)
)

# True-color vis params (bands are scaled ×10 000 in S2 SR)
s2_vis = {"bands": ["B4", "B3", "B2"], "min": 0, "max": 3000}

## GEDI L2A Monthly (pre-tiled raster from LARSE)

In [6]:
# LARSE/GEDI/GEDI02_A_002_MONTHLY — pre-tiled, easier for pixel-level extraction
gedi = (
    ee.ImageCollection("LARSE/GEDI/GEDI02_A_002_MONTHLY")
    .filterBounds(california)
    .filterDate("2025-01-01", "2026-04-09")
    .select("rh98")  # relative height at 98th percentile ≈ canopy top height
    .median()
    .clip(california)
)

gedi_vis = {"min": 0, "max": 40, "palette": ["#ffffcc", "#41b6c4", "#253494"]}

## Interactive Map — Sentinel-2 + GEDI L2A over California

In [7]:
Map = geemap.Map(center=[37.5, -119.5], zoom=6)

Map.addLayer(s2, s2_vis, "Sentinel-2 True Color")
Map.addLayer(gedi, gedi_vis, "GEDI L2A rh98 (canopy height, m)")

Map.add_colorbar(gedi_vis, label="GEDI rh98 canopy height (m)")
Map.addLayerControl()
Map

Map(center=[37.5, -119.5], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright'…